# Explanations: what the app will say about each lounge and area

Each decision (`decisions.ipynb`) gets a pyramid: a one-sentence answer, then the arguments that
matter most for it (ranked by `src.explain.prioritize`, never by the writer), each backed by 2-5
numbers copied from the decision's fact sheet. `src.explain.verify` rejects any explanation that
cites a value or writes a number not on the fact sheet.

The **AI** explanations here were written in a Claude Code session from the same prompts
`just explain` sends to the API (`python -m src.explain prompts|check|ingest`), so no API credits
were spent; they are cached in `data/explanations/cache.json` by a hash of their fact sheet, so a
changed number falls back to the deterministic **template**. The NOT SCORED airport lounge and
SKIP areas always use the template.

Data: `src.explain.v3_subjects()` (baseline, medium levels).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from collections import Counter

import pandas as pd
from IPython.display import Markdown, display

from src.explain import (LOUNGE_TABLE, AREA_TABLE, explain, load_cache, table_rows,
                         v3_subjects, verify)

pd.set_option("display.max_colwidth", 120)
cache = load_cache()
subjects = v3_subjects()
exps = {(k, sid): explain(k, sid, a, f, cache=cache) for k, sid, a, f in subjects}
facts = {(k, sid): f for k, sid, a, f in subjects}
print(Counter((k, e.source) for (k, _), e in exps.items()))
print("grounding failures:", sum(bool(verify(e, facts[key], key[0])) for key, e in exps.items()))

Counter({('area', 'template'): 539, ('area', 'ai'): 41, ('lounge', 'ai'): 23, ('uae', 'ai'): 1, ('lounge', 'template'): 1})
grounding failures: 0


In [2]:
def show(kind, sid):
    e, f = exps[(kind, sid)], facts[(kind, sid)]
    lines = [f"### {sid} — **{e.action}** ({e.source})", "", f"**{e.headline}**", ""]
    for r in e.reasons:
        ev = "; ".join(f"{x.label}: {x.value}" for x in r.evidence)
        lines += [f"- **{r.topic}**: {r.claim}  \n  <small>{ev}</small>"]
    lines += ["", f"_{e.table_caption}_"]
    display(Markdown("\n".join(lines)))


def table(kind, sid):
    fields = {"lounge": LOUNGE_TABLE, "area": AREA_TABLE}[kind]
    return pd.DataFrame(table_rows(fields, facts[(kind, sid)])).set_index("Factor")

## The network summary (the app opens with this)

In [3]:
show("uae", "uae")

### uae — **SUMMARY** (ai)

**Protect three lounges, investigate seven, and open in four Sharjah-area growth areas that together hold 218,300 women beyond every lounge's reach.**

- **shrink**: Seven of the 23 scored lounges (al-maqta, delma, khaleej-al-arabi, mohammed-bin-zayed-city, noya-plaza, shahama, westyas) are weak on most signals, with composites of 0.33 or lower against the 0.35 bar, and are the first to investigate: weak market position, not proven losses.  
  <small>SHRINK: 7; Lounges scored: 23; SHRINK lounges: al-maqta, delma, khaleej-al-arabi, mohammed-bin-zayed-city, noya-plaza, shahama, westyas; Strongest SHRINK composite: 0.3266</small>
- **grow**: Four areas pass both tests (al-dhaid-sharjah, kalba-sharjah, khor-fakkan-sharjah, sharjah-sharjah), covering 218,300 women 15+ between 11.1 and 43.8 km from the nearest lounge, and none is more crowded than 44.1 premium reviews per 1,000 women; a further 37 areas are on WATCH.  
  <small>GROW: 4; Women in GROW areas: 218300; Closest GROW area to a branch: 11.0963; Furthest GROW area from a branch: 43.7824; Most saturated GROW area: 44.0921</small>
- **protect**: Three lounges (al-ain, al-taif-mall, ras-al-khaimah) clear the 0.65 PROTECT bar, the weakest at 0.66, so they are strong on most signals and are the base to defend while the network is reshaped.  
  <small>PROTECT: 3; PROTECT lounges: al-ain, al-taif-mall, ras-al-khaimah; Weakest PROTECT composite: 0.6643</small>
- **confidence**: The middle is large: 13 lounges sit in HOLD and 10 of the 23 calls are low confidence, because they are close to a threshold, missing an input or flip when assumptions change, so treat the lists as a shortlist to verify.  
  <small>HOLD: 13; Low-confidence calls: 10; Lounges scored: 23; Not scored: 1</small>
- **data_gaps**: We have no revenue or rent data, so these calls show where demand and position look strong or weak across 3,833 salons searched, not what any lounge earns or costs.  
  <small>Revenue data: False; Rent data: False; Salons searched: 3833</small>

_Each lounge gets a composite score between 0 and 1: 0.65 or more is PROTECT, 0.35 or less is SHRINK, and everything between is HOLD. Growth areas are populated places beyond a 15-min drive of every lounge, and GROW needs both enough women (20,000 or more) and light premium competition. Without revenue or rent data, read the calls as signals to test with the business, not as financial results._

## Lounges, PROTECT first

In [4]:
order = {"PROTECT": 0, "HOLD": 1, "SHRINK": 2, "NOT SCORED": 3}
for k, sid, a, f in sorted((s for s in subjects if s[0] == "lounge"), key=lambda s: (order[s[2]], s[1])):
    show(k, sid)

### al-ain — **PROTECT** (ai)

**Protect Al Ain: it has its market to itself and wins a clear share of a deep premium scene, so it is the kind of branch to defend, with the call holding under every assumption tested.**

- **cannibalisation**: No other Bedashing lounge reaches the women Al Ain serves, so every customer it wins is new to the network rather than moved from a sibling.  
  <small>Shared catchment: 0.0; Cannibalisation score: 1.0; Women 15+ in catchment: 102438.9547</small>
- **capture**: Against 16 serious premium rivals it still takes about 12% of premium reviews, a strong position for a market with 59 premium salons.  
  <small>Capture: 0.1232; Premium substitutes: 16; Premium salons in catchment: 59; Capture score: 0.8211</small>

_Read the cannibalisation row first: a shared catchment of zero means no sibling is competing for these women. Then check capture, which shows how much of the premium review pool the lounge wins; a high score here despite many rivals is why this call is not fragile._

### al-taif-mall — **PROTECT** (ai)

**Protect Al Taif Mall: it has its catchment to itself and takes a large share of a mid-sized premium market, which outweighs a small local population.**

- **cannibalisation**: No sibling lounge reaches its customers, so its results are not borrowed from other branches.  
  <small>Shared catchment: 0.0; Cannibalisation score: 1.0</small>
- **capture**: Against 7 serious rivals it holds about 14% of premium reviews, a standout share that earns a near-perfect capture score.  
  <small>Capture: 0.1428; Premium substitutes: 7; Premium salons in catchment: 35; Capture score: 0.9523</small>

_Demand is the weak row here, about 45,000 women, so the lounge wins on competitive position rather than market size. The composite of 0.69 clears the 0.65 bar, and the call does not change under any assumption tested._

### ras-al-khaimah — **PROTECT** (ai)

**Protect Ras Al Khaimah: it has its market to itself and takes a strong share of the premium scene, though the margin over the bar is thin.**

- **cannibalisation**: No sibling lounge reaches its customers, so everything it earns is new to the network.  
  <small>Shared catchment: 0.0; Cannibalisation score: 1.0; Addressable women 15+: 55839.6964</small>
- **capture**: Against 8 serious rivals it holds about 14% of premium reviews, a standout position that more than makes up for a modest market.  
  <small>Capture: 0.1381; Premium substitutes: 8; Premium salons in catchment: 44; Capture score: 0.9209</small>

_The composite of 0.66 only just clears the 0.65 bar, and the rating row is weak since it trails its rivals. Demand is unweighted because no rents are observed outside Dubai, and the call changes in 18 of 81 assumption combinations, so it is a fair but not rock-solid PROTECT._

### al-barsha — **HOLD** (ai)

**Hold Al Barsha: rent-lifted demand is the top score, but it shares most of its women with sibling lounges and wins a tiny slice of a crowded premium scene, and the call flips in 27 of 81 assumption combinations, so confidence is low.**

- **demand**: Observed rents lift the roughly 272,000 catchment women to about 321,000 addressable women, the top demand score, though that only matters if the lounge can win them.  
  <small>Women 15+ in catchment: 272370.6414; Addressable women 15+: 320730.5673; Affluence coverage: 0.9899; Demand score: 1.0</small>
- **capture**: Against 63 serious rivals in a pool of 228 premium salons, Al Barsha captures only about 1% of premium reviews, so its size advantage is not converting.  
  <small>Capture: 0.0081; Premium substitutes: 63; Premium salons in catchment: 228; Capture score: 0.0542</small>
- **cannibalisation**: About 86% of its women are also reached by another Bedashing lounge, so growing here largely takes customers from our own branches.  
  <small>Shared catchment: 0.8553; Cannibalisation score: 0.1447</small>
- **rating**: Its 4.6 stars trail the rivals' median of 4.8, so quality is not offsetting the crowding.  
  <small>Google rating: 4.6; Substitutes' median rating: 4.8; Rating gap: -0.2; Rating score: 0.1667</small>

_Demand is maxed out here, helped by a rent weighting that covers 99% of the women, so look past it to capture and cannibalisation, where the lounge scores poorly. The composite of 0.37 sits barely above the 0.35 SHRINK line and the call flips in 27 of 81 combinations, so treat it as low confidence._

### al-dhafra — **HOLD** (ai)

**Hold Al Dhafra: it is alone in a small market, which keeps it clear of sibling overlap but caps its upside, and a thin premium market makes its strong share a weak signal.**

- **cannibalisation**: No sibling lounge reaches its customers, so there is no internal competition to worry about.  
  <small>Shared catchment: 0.0; Cannibalisation score: 1.0</small>
- **demand**: The catchment holds only about 23,000 women, unweighted by affluence because no rents are observed outside Dubai, so the demand score is low.  
  <small>Addressable women 15+: 22927.3037; Women 15+ in catchment: 22927.3037; Affluence coverage: 0.0; Demand score: 0.1146</small>
- **rating**: At 4.5 stars it trails the premium rivals' median of 4.7, so quality does not lift the case.  
  <small>Google rating: 4.5; Substitutes' median rating: 4.7; Rating gap: -0.2; Rating score: 0.1667</small>

_Only 7 premium salons exist in the catchment, so the capture figure is noise and the model scores it neutral at 0.5; do not read the high capture percentage as a win. The call rests on zero overlap against small demand and a below-rival rating, and it holds under every assumption tested._

...

### mohammed-bin-zayed-city — **SHRINK** (ai)

**Investigate Mohammed Bin Zayed City: all its women are shared with sibling lounges, it wins a small share of the premium market and trails rivals on rating, but the call flips in 54 of 81 assumption combinations, so confidence is low.**

- **cannibalisation**: Its whole catchment overlaps with other Bedashing lounges, so another lounge competes for every woman it serves (the model can't say where they would go if it closed).  
  <small>Shared catchment: 1.0; Cannibalisation score: 0.0; Women 15+ in catchment: 129055.6898</small>
- **capture**: Against 18 serious rivals in a pool of 80 premium salons it holds only about 4% of premium reviews.  
  <small>Capture: 0.0361; Premium substitutes: 18; Premium salons in catchment: 80; Capture score: 0.2407</small>
- **rating**: At 4.4 stars it trails the rivals' median of 4.6, so quality does not offset the weak share.  
  <small>Google rating: 4.4; Substitutes' median rating: 4.6; Rating gap: -0.2; Rating score: 0.1667</small>

_Every signal except demand is weak, giving a composite of 0.28, and demand is unweighted because no rents are observed outside Dubai. But 54 flips of 81 mean the answer depends more on the assumptions than on the data, so treat this as a branch to investigate rather than a settled verdict._

### noya-plaza — **SHRINK** (ai)

**Investigate Noya Plaza: every woman it reaches is also reached by a sibling lounge, the local market is small and its premium share is tiny, a clear and stable call.**

- **cannibalisation**: Its whole catchment overlaps with other Bedashing lounges, so another lounge competes for every woman it serves (the model can't say where they would go if it closed).  
  <small>Shared catchment: 1.0; Cannibalisation score: 0.0</small>
- **capture**: Against 9 serious rivals it holds only about 3% of premium reviews, and its review base is small.  
  <small>Capture: 0.0272; Premium substitutes: 9; Google reviews: 215; Capture score: 0.1811</small>
- **demand**: Only about 45,000 women live in reach, unweighted by affluence because no rents are observed outside Dubai, so there is little market to grow into.  
  <small>Addressable women 15+: 45036.5814; Affluence coverage: 0.0; Cells in catchment: 27; Demand score: 0.2252</small>

_Its 4.7 rating is its one good number, but it cannot offset overlap, low share and a small market. The composite of 0.21 is well below the 0.35 line, and the call does not change under any assumption tested._

### shahama — **SHRINK** (ai)

**Investigate Shahama: sibling lounges reach most of its women, the local market is small and it trails rivals on rating, but the call flips in 63 of 81 assumption combinations, so confidence is low.**

- **cannibalisation**: About 87% of its women are also served by another Bedashing lounge, so another lounge competes for every woman it serves (the model can't say where they would go if it closed).  
  <small>Shared catchment: 0.8697; Cannibalisation score: 0.1303</small>
- **demand**: Only about 66,000 women live in reach, unweighted by affluence because no rents are observed outside Dubai, a modest base for a branch that has to defend itself against overlap.  
  <small>Addressable women 15+: 65769.1568; Affluence coverage: 0.0; Cells in catchment: 40; Demand score: 0.3288</small>
- **rating**: At 4.6 stars it trails the rivals' median of 4.8, so quality does not offset the weaker signals.  
  <small>Google rating: 4.6; Substitutes' median rating: 4.8; Rating gap: -0.2; Rating score: 0.1667</small>

_Its capture score is the one decent row. The composite of 0.32 is only just under the 0.35 line, and with 63 flips of 81 the SHRINK label depends more on the assumptions than on the data, so investigate before acting._

### westyas — **SHRINK** (ai)

**Investigate Westyas: nearly all its women are shared with sibling lounges, the local market is small and it trails rivals on rating, a clear and stable call.**

- **cannibalisation**: About 98% of its catchment overlaps with other Bedashing lounges, so the branch adds almost nothing the network does not already reach.  
  <small>Shared catchment: 0.9823; Cannibalisation score: 0.0177</small>
- **demand**: Only about 42,000 women live in reach, unweighted by affluence because no rents are observed outside Dubai, so there is little room to grow.  
  <small>Addressable women 15+: 42368.0713; Affluence coverage: 0.0; Cells in catchment: 27; Demand score: 0.2118</small>
- **rating**: At 4.6 stars it trails the rivals' median of 4.8, so quality gives no reason to keep it as is.  
  <small>Google rating: 4.6; Substitutes' median rating: 4.8; Rating gap: -0.2; Rating score: 0.1667</small>

_Three of the four signals are weak, giving a composite of 0.22 well below the 0.35 line. The call does not change under any assumption tested, so it is a confident one._

### zayed-international-airport — **NOT SCORED** (template)

**Not scored: this lounge serves travellers, not the women living around it, so the catchment model doesn't apply.**

- **demand**: Addressable women 15+: 75,830 women, shown but not scored.  
  <small>Addressable women 15+: 75830.1549; Women 15+ in catchment: 75830.1549; Affluence coverage: 0.0; Women captured (est.): 0.0</small>
- **capture**: Capture: 4% of catchment, shown but not scored.  
  <small>Capture: 0.0392; Premium substitutes: 11; Premium salons in catchment: 51; Thin premium market: False</small>

_Each signal is scored 0-1 on a fixed scale, where 1 is good for the lounge, and the composite is their weighted average. Demand counts women within a 15-min drive, weighted by affluence where Dubai rents are known, cannibalisation how many of them another lounge also reaches, capture the lounge's share of premium-salon reviews nearby, and rating its Google rating against those rivals._

The factor table the app shows next to a lounge's pyramid, for one example:

In [5]:
table("lounge", "al-barsha")

,Value,Unit,Threshold,What it means
Factor,,,,
Women 15+ in catchment,"272,371",women,—,"Women aged 15+ living in the ~2 km cells within a 15-min drive (typical midday traffic). WorldPop adults, rebalanced..."
Addressable women 15+,"320,731",women,"GROW needs 20,000","Catchment women weighted by how affluent their cell is (observed median rent vs the women-weighted median, to the af..."
Affluence rent,"115,000",AED/yr,—,"Median annual household rent (DLD Ejari contracts, Jul-Oct 2026), weighted by women, over the cells with an observed..."
Affluence coverage,99%,% of women,"Under 50%: affluence mostly unknown, weighted neutral",Share of the women here living in cells with an observed rent. The rest are weighted neutral: affluence unknown. Dub...
Cells in catchment,64,count,—,Populated ~2 km grid cells whose centre is within the drive time.
Shared catchment,86%,% of catchment women,"Cannibalisation score: 100% of catchment → 0, 0% of catchment → 1; weight 1",Share of the catchment's women who live in cells another open Bedashing lounge also reaches: the cannibalisation mea...
Premium salons in catchment,228,count,—,"Google Places salons in the catchment that count as premium: Google price 'expensive' or above, or (unpriced) at lea..."
Premium substitutes,63,count,—,The most-reviewed premium salons that together hold 60% of the catchment's premium reviews: the lounge's real compet...
Capture,1%,% of reviews,"Capture score: 0% of catchment → 0, 15% of catchment → 1; weight 1","Lounge reviews ÷ (lounge + substitutes' reviews, scaled up for salons the search missed). A proxy for share of premi..."


## Growth areas: GROW, then WATCH by women

In [6]:
areas = sorted((s for s in subjects if s[0] == "area" and s[2] != "SKIP"),
               key=lambda s: (s[2] != "GROW", -s[3]["women"]))
for k, sid, a, f in areas:
    show(k, sid)

### sharjah-sharjah — **GROW** (ai)

**GROW Sharjah: 90,606 women, over four times the 20,000 line, and a premium market well under the 50-review line.**

- **size**: Sharjah has 90,606 women 15+, more than four times the 20,000 line, with only a small share in worker housing.  
  <small>Women 15+: 90605.7873; Addressable women 15+: 90605.7873; Size line: 20000; Big enough: True</small>
- **saturation**: Its 15 premium salons draw 21.9 reviews per 1,000 women, under half the 50 line, so there is room alongside existing competition.  
  <small>Premium salons: 15; Premium saturation: 21.9345; Saturation line: 50.0; Unsaturated: True</small>
- **reach**: The nearest lounge, al-jada, is 13.3 km away in a straight line and beyond a 15-min drive, so a Sharjah lounge would sit at the edge of the network alongside Bedashing's two Sharjah-emirate lounges, which are near the Dubai border.  
  <small>Nearest lounge: al-jada; Distance to nearest lounge: 13.2622</small>

_Women 15+ is the local customer pool; a lounge needs 20,000. Premium reviews per 1,000 women show how crowded the premium market is, and under 50 counts as open. Sharjah passes both tests by a wide margin, which is why it is a GROW._

### al-dhaid-sharjah — **GROW** (ai)

**GROW: Al Dhaid, Sharjah is a 73,067-woman market with light competition, so Bedashing should look at opening here.**

- **size**: Al Dhaid has 73,067 addressable women 15+, more than three times the 20,000 line.  
  <small>Addressable women 15+: 73067.0302; Size line: 20000; Big enough: True</small>
- **saturation**: Eight premium salons yield only 5.4 premium reviews per 1,000 women, about a tenth of the line of 50, so demand looks under-served.  
  <small>Premium salons: 8; Premium saturation: 5.4338; Saturation line: 50.0</small>
- **reach**: The nearest lounge, al-jada, is 43.8 km away, so a lounge here would be a standalone market and a new hub rather than an extension of the network.  
  <small>Nearest lounge: al-jada; Distance to nearest lounge: 43.7824</small>

_The table shows how many women 15+ live in Al Dhaid, how crowded premium salons are per 1,000 women (under 50 counts as open), and how far the nearest lounge is. Al Dhaid passes both growth tests by a wide margin; the distance means it would need to stand on its own._

### khor-fakkan-sharjah — **GROW** (ai)

**GROW Khor Fakkan: 31,163 women and a premium market well under the 50-review line, in a market far from any lounge.**

- **size**: Khor Fakkan has 31,163 women 15+, well above the 20,000 line, with almost no worker housing.  
  <small>Women 15+: 31162.5661; Addressable women 15+: 31162.5661; Size line: 20000; Big enough: True</small>
- **saturation**: Its 11 premium salons draw 28.5 reviews per 1,000 women, under the 50 line, so there is room alongside existing competition.  
  <small>Premium salons: 11; Premium saturation: 28.4775; Saturation line: 50.0; Unsaturated: True</small>
- **reach**: The nearest lounge, al-taif-mall, is 26.5 km away in a straight line, so Khor Fakkan would be a standalone market.  
  <small>Nearest lounge: al-taif-mall; Distance to nearest lounge: 26.5027</small>

_Women 15+ is the local customer pool; a lounge needs 20,000. Premium reviews per 1,000 women show how crowded the premium market is, and under 50 counts as open. Khor Fakkan passes both tests, which is why it is a GROW._

### kalba-sharjah — **GROW** (ai)

**GROW Kalba: 23,464 women and a premium market under the 50-review line make it a lounge candidate.**

- **size**: Kalba has 23,464 women 15+, above the 20,000 line, with almost no worker housing to cloud the estimate.  
  <small>Women 15+: 23464.2586; Addressable women 15+: 23464.2586; Size line: 20000; Big enough: True</small>
- **saturation**: Its 6 premium salons draw 44.1 reviews per 1,000 women, under the 50 line but not by a wide margin, so competition exists and the gap is narrower than the headline suggests.  
  <small>Premium salons: 6; Premium saturation: 44.0921; Saturation line: 50.0; Unsaturated: True</small>
- **reach**: The nearest lounge, al-taif-mall, is 11.1 km away in a straight line and beyond a 15-min drive, so Kalba would sit near the edge of the network and serve women that lounge does not reach.  
  <small>Nearest lounge: al-taif-mall; Distance to nearest lounge: 11.0963</small>

_Women 15+ is the local customer pool; a lounge needs 20,000. Premium reviews per 1,000 women show how crowded the premium market is, and under 50 counts as open. Kalba passes both tests, which is why it is a GROW._

### al-madam-sharjah — **WATCH** (ai)

**WATCH: Al Madam, Sharjah is worth watching because it is big and looks unserved, but we searched only 42% of its women, so verify before opening.**

- **size**: Al Madam has 33,679 addressable women 15+, well over the 20,000 line.  
  <small>Addressable women 15+: 33678.5487; Size line: 20000; Big enough: True</small>
- **saturation**: One premium salon and 1.1 premium reviews per 1,000 women look almost empty against a line of 50.  
  <small>Premium salons: 1; Premium saturation: 1.0507; Saturation line: 50.0</small>
- **reach**: The nearest lounge, mirdif-35, is 47.4 km away, so this would be a fully standalone market.  
  <small>Nearest lounge: mirdif-35; Distance to nearest lounge: 47.3514</small>
- **data_gap**: Our search reached only 42% of the area's women against the 50% needed, so the near-zero competition could be a data hole and the call stays at WATCH.  
  <small>Competitor data coverage: 0.4239; Coverage needed: 0.5; Unsaturated: True</small>

_The table shows the women 15+ in Al Madam, how crowded premium salons are per 1,000 women, the distance to our nearest lounge and how much of the area our salon search covered. The size is a clear pass, but the competition figure rests on less than half the area, so it needs a wider search before anyone acts._

...

### al-bihouth-abu-dhabi — **WATCH** (ai)

**WATCH: Al Bihouth, Abu Dhabi is worth watching because no premium rival was found, but at 5,255 women it is far too small to open yet.**

- **size**: Al Bihouth has 5,255 addressable women 15+, only just above the 5,000 floor and about a quarter of the 20,000 line.  
  <small>Addressable women 15+: 5255.2347; Size line: 20000; Size floor: 5000</small>
- **saturation**: We found no premium salons and zero premium reviews per 1,000 women, so there is no rival to fight, though that also gives no proof of demand.  
  <small>Premium salons: 0; Premium saturation: 0.0; Saturation line: 50.0</small>
- **reach**: The nearest lounge, baniyas, is 10.4 km away, so a site here would sit near the edge of our Abu Dhabi network, easier to run, though beyond the lounge's catchment.  
  <small>Nearest lounge: baniyas; Distance to nearest lounge: 10.402</small>

_The table shows how many women 15+ live in Al Bihouth, how crowded premium salons are per 1,000 women (under 50 counts as open) and how far the nearest lounge is. Competition is a clear pass, but the population sits right above the 5,000 floor, which is the weak point._

### al-marmoom-dubai — **WATCH** (ai)

**WATCH: Al Marmoom, Dubai is worth watching only for its empty competitor field; at 5,118 addressable women it is too thin to open.**

- **size**: Al Marmoom has 5,118 addressable women 15+, only just above the 5,000 floor.  
  <small>Addressable women 15+: 5118.3907; Women 15+: 5187.2713; Size line: 20000; Size floor: 5000</small>
- **saturation**: No premium salons and zero premium reviews per 1,000 women, though our search covered only 52% of the area's women.  
  <small>Premium salons: 0; Premium saturation: 0.0; Competitor data coverage: 0.5196</small>
- **reach**: The nearest lounge, nad-al-sheba, is 22.7 km away, so this would be a standalone market.  
  <small>Nearest lounge: nad-al-sheba; Distance to nearest lounge: 22.7169</small>

_The table shows the addressable women 15+ in Al Marmoom, how crowded premium salons are per 1,000 women, and how much of the area our search covered. Competition looks empty, but the population is close to the 5,000 floor under which an area is dropped altogether._

### manama-ajman — **WATCH** (ai)

**WATCH Manama: virtually no premium competition, but at 5,147 women it is far too small for a lounge today.**

- **size**: Manama has 5,147 women 15+, just above the 5,000 floor and about a quarter of the 20,000 line.  
  <small>Women 15+: 5146.8426; Addressable women 15+: 5146.8426; Size line: 20000; Size floor: 5000; Big enough: False</small>
- **saturation**: Its 1 premium salon draws 0.4 reviews per 1,000 women, far under the 50 line, so the premium market is essentially open.  
  <small>Premium salons: 1; Premium saturation: 0.3886; Saturation line: 50.0; Unsaturated: True</small>
- **reach**: The nearest lounge, al-taif-mall, is 39.3 km away in a straight line, so Manama would be a standalone market.  
  <small>Nearest lounge: al-taif-mall; Distance to nearest lounge: 39.3036</small>

_Women 15+ is the local customer pool; a lounge needs 20,000 and anything under 5,000 is skipped. Premium reviews per 1,000 women show how crowded the premium market is. Manama passes competition and misses size, so it is a WATCH._

### dubai-dubai — **WATCH** (ai)

**WATCH: this Dubai pocket is worth watching only because it is uncontested; with 5,108 women, barely above the floor, it is not yet an opening candidate.**

- **size**: This Dubai area has 5,108 addressable women 15+, barely above the 5,000 floor.  
  <small>Addressable women 15+: 5108.1128; Size line: 20000; Size floor: 5000</small>
- **saturation**: No premium salons and zero premium reviews per 1,000 women, against a line of 50.  
  <small>Premium salons: 0; Premium saturation: 0.0; Saturation line: 50.0</small>
- **reach**: The nearest lounge, mirdif-35, is 12.5 km away, so a site here would sit near the edge of our Dubai network, easier to run, though beyond the lounge's catchment.  
  <small>Nearest lounge: mirdif-35; Distance to nearest lounge: 12.5468</small>

_The table shows the women 15+ in this Dubai area, how crowded premium salons are per 1,000 women, and the distance to our nearest lounge. The area passes on competition but is among the smallest we track, only just above the 5,000 floor._

### khatt-ras-al-khaimah — **WATCH** (ai)

**WATCH Khatt: only 5,073 women and our salon search covered a quarter of them, so the competition read is unreliable.**

- **size**: Khatt has 5,073 women 15+, barely above the 5,000 floor and about a quarter of the 20,000 line.  
  <small>Women 15+: 5073.2806; Addressable women 15+: 5073.2806; Size line: 20000; Size floor: 5000; Big enough: False</small>
- **saturation**: No premium salons were found and reviews per 1,000 women are 0.0, but that rests on a thin search.  
  <small>Premium salons: 0; Premium saturation: 0.0; Saturation line: 50.0; Unsaturated: True</small>
- **reach**: The nearest lounge, ras-al-khaimah, is 14.9 km away in a straight line and beyond a 15-min drive, so Khatt would sit near the edge of the network.  
  <small>Nearest lounge: ras-al-khaimah; Distance to nearest lounge: 14.8786</small>
- **data_gap**: Our salon search covered only 25% of Khatt's women against the 50% needed, so the open-market reading cannot be trusted yet.  
  <small>Competitor data coverage: 0.2496; Coverage needed: 0.5; Unsaturated: True</small>

_Women 15+ is the local customer pool; a lounge needs 20,000 and anything under 5,000 is skipped. Competitor coverage shows how much of the area our salon search actually saw. Khatt is small and under-searched, so it is a WATCH._

In [7]:
table("area", areas[0][1])

,Value,Unit,Threshold,What it means
Factor,,,,
Women 15+,"90,606",women,"SKIP under 5,000",Women aged 15+ living in the area's cells (raw).
Addressable women 15+,"90,606",women,"GROW needs 20,000","Catchment women weighted by how affluent their cell is (observed median rent vs the women-weighted median, to the af..."
Affluence rent,n/a (missing),AED/yr,—,"Median annual household rent (DLD Ejari contracts, Jul-Oct 2026), weighted by women, over the cells with an observed..."
Affluence coverage,0%,% of women,"Under 50%: affluence mostly unknown, weighted neutral",Share of the women here living in cells with an observed rent. The rest are weighted neutral: affluence unknown. Dub...
Cells,50,count,—,Populated ~2 km cells in the area (one contiguous piece).
Worker housing,11%,% of adults,GROW needs under 50%,Share of the area's adults living in worker housing (OSM industrial land use).
Premium salons,15,count,—,Premium salons found in the area's searched cells.
Premium saturation,21.93,reviews per 1k women,Unsaturated under 50,"Premium salons' Google reviews (scaled up for missed salons) per 1,000 women in the searched cells. Lounge catchment..."
Competitor data coverage,96%,% of women,GROW needs 50%,Share of the area's women in cells our salon search covered.


## Play with it

Change a number and see the fallback: the cache key is a hash of the fact sheet, so any change
means the template is served instead of the cached AI text (and `verify` would reject the old
text anyway).

In [8]:
k, sid, a, f = next(s for s in subjects if s[1] == "al-barsha")
changed = {**f, "catchment_women": f["catchment_women"] + 1}
print(explain(k, sid, a, changed, cache=cache).source, "<- template, because one number changed")

template <- template, because one number changed
